In [ ]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

In [ ]:
with open('TinyShakespeare.txt', encoding='UTF-8') as w:
    text = w.read()

In [ ]:
vocabulary = sorted(list(set(text)))
encode = lambda x: torch.Tensor([vocabulary.index(value) for value in x]).to(dtype=torch.long).to('cuda')
decode = lambda x: "".join([vocabulary[value] for value in x])
text = encode(text)
n = int(.9*len(text))
train_data = text[:n]
val_data = text[n:]
batch_size = 64
block_size = 256
torch.manual_seed(1337)
def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data)- block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

batch = get_batch('train')

# B, T, C = batch[0].shape
# batch[0].view(B*T, C)

In [ ]:
# print(batch[0][1])
# print(batch[1][1])

In [ ]:
sequenceLength = 8

In [ ]:
# print(''.join(map(str, decode(text))))

In [ ]:


torch.manual_seed(1337)
class ZachEmbedding(nn.Module):
    def __init__(self, embedding_dim, num_embeddings):
        super(ZachEmbedding, self).__init__()
        self.weights = nn.Parameter(data=torch.randn(num_embeddings, embedding_dim))
    def forward(self, input):
        if not isinstance(input, torch.LongTensor):
            input = input.long()
        return self.weights[input]
class ZachHead(nn.Module):
    def __init__(self, head_size):
        super(ZachHead, self).__init__()
        self.key = Algos.ZachLinear(n_embd, head_size, bias=False)
        self.query = Algos.ZachLinear(n_embd, head_size, bias=False)
        self.value = Algos.ZachLinear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    def forward(self, x):
        B, T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * C**-.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out
class ZachMultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        self.neadsize = head_size
        super().__init__()
        self.heads = nn.ModuleList([ZachHead(head_size) for _ in range(num_heads)])
        self.proj = Algos.ZachLinear(n_embd, n_embd)
    def forward(self, x):
        rawOutputs = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(rawOutputs)
class ZachTransformerFeedForward(nn.Module):
    def __init__(self,n_embd):
        super().__init__()
        self.net = nn.Sequential(
            Algos.ZachLinear(n_embd, 4 * n_embd),
            nn.ReLU(),
            Algos.ZachLinear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )
    def forward(self,x):
        return self.net(x)
class ZachTransformmerBlock(nn.Module):
    def __init__(self,n_embd, n_heads):
        super().__init__()
        headsize = n_embd//n_heads
        self.sa_heads = ZachMultiHeadAttention(n_heads, headsize)
        self.ffwd = ZachTransformerFeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa_heads(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x
class TinyShakespeareThing(nn.Module):
    def __init__(self):
        super(TinyShakespeareThing, self).__init__()
        self.embedding = ZachEmbedding(num_embeddings=len(vocabulary), embedding_dim=n_embd)
        self.positionalEncoding = ZachEmbedding(num_embeddings=block_size, embedding_dim=n_embd)
        self.blocks = nn.Sequential(*[ZachTransformmerBlock(n_embd, n_heads=n_head) for _ in range(n_layer)], nn.LayerNorm(n_embd))
        self.lm_head = Algos.ZachLinear(in_features=n_embd, out_features=len(vocabulary))
    def forward(self, x, target=None):
        B, T = x.shape
        tok_emb = self.embedding(x)
        posEncoding = self.positionalEncoding(torch.arange(T, device='cuda'))
        wencoding = tok_emb + posEncoding
        wencoding = self.blocks(wencoding)
        logits = self.lm_head(wencoding)
        if target is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits, target)
        return logits, loss
    def generate(self, idx, max_tokens):
        for _ in range(max_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx.to('cuda'), idx_next.to('cuda')), dim=1)
        return idx

In [ ]:
model = TinyShakespeareThing().to('cuda')
batch = get_batch("train")
result = model(x=batch[0], target=batch[1])
print('LOSS:')
print(result[1])
print('GENERATION:')
print(decode(model.generate(torch.zeros((1,1), dtype=torch.long).to('cuda'), max_tokens=100)[0].tolist()))

In [ ]:
batch[1].dtype

In [ ]:

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, y= get_batch(split)
            logits ,loss = model(X,y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
# print('beas')
from tqdm import tqdm
epochs = 5000
eval_interval = 500
eval_iters = 200
optimizer = torch.optim.AdamW(params=model.parameters(), lr=3e-4)
for epoch in tqdm(range(epochs)):
    model.train()
    loss = model(*get_batch('train'))[1]
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if(epoch%eval_iters == 0):
        losses = estimate_loss()
        print(f"Epoch: {epoch}, Train Loss: {losses['train']:.4f}, Val loss: {losses['val']:.4f}")

In [ ]:
print(decode(model.generate(torch.zeros((1,1), dtype=torch.long).to('cuda'), max_tokens=400)[0].tolist()))

In [ ]:
print(decode(model.generate(torch.zeros((1,1), dtype=torch.long).to('cuda'), max_tokens=400)[0].tolist()))

In [ ]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import Algos
from tqdm import tqdm


### HyperParameters
batch_size = 64
block_size = 256
n_embd = 384
dropout = .2
n_head = 6
n_layer = 6
eval_iters= 200
epochs = 5000
eval_interval = 500
learning_rate = 3e-4

### Seed Setting

torch.manual_seed(1337)
torch.cuda.manual_seed(1337)

### Data Processing
with open('TinyShakespeare.txt', encoding='UTF-8') as w:
    text = w.read()

vocabulary = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(vocabulary)}
itos = {i: ch for i, ch in enumerate(vocabulary)}
encode = lambda x: torch.tensor([stoi[y] for y in x], dtype=torch.long).to('cuda')
decode = lambda x: ''.join([itos[y] for y in x])
text = encode(text)
n = int(.9*len(text))
train_data = text[:n]
val_data = text[n:]
torch.manual_seed(1337)
def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data)- block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

In [ ]:
%%timeit
get_batch('train')